# Statements Examples

This notebook mirrors `examples/scripts/statements/statements_example.py`. It walks through the same five scenarios to demonstrate the Python bindings for the statements engine:

1. Build a basic P&L model with actuals and forecasts.
2. Evaluate a model and inspect node values period by period.
3. Explore the dynamic metric registry.
4. Initialize the extension system.
5. Serialize and deserialize a model with JSON.

Run the cells sequentially to reproduce the CLI example outputs inline.


In [7]:
from finstack.core.dates.periods import PeriodId
from finstack.statements.builder import ModelBuilder
from finstack.statements.evaluator import Evaluator
from finstack.statements.extensions import ExtensionRegistry
from finstack.statements.registry import Registry
from finstack.statements.types import AmountOrScalar, ForecastSpec



## Example 1 – Basic P&L Model

Construct a quarterly P&L with actuals for Q1–Q2 2025 and forecast formulas for the remaining periods. We also inspect the resulting periods and metadata on the built model.


In [8]:
builder = ModelBuilder.new("Acme Corp Q1-Q4 2025")
builder.periods("2025Q1..Q4", "2025Q2")  # Q1-Q2 actuals, Q3-Q4 forecast

builder.value(
    "revenue",
    [
        (PeriodId.quarter(2025, 1), AmountOrScalar.scalar(10_000_000.0)),
        (PeriodId.quarter(2025, 2), AmountOrScalar.scalar(11_000_000.0)),
    ],
)

builder.value(
    "operating_expenses",
    [
        (PeriodId.quarter(2025, 1), AmountOrScalar.scalar(2_000_000.0)),
        (PeriodId.quarter(2025, 2), AmountOrScalar.scalar(2_100_000.0)),
    ],
)

builder.compute("cogs", "revenue * 0.6")
builder.compute("gross_profit", "revenue - cogs")
builder.compute("operating_income", "gross_profit - operating_expenses")
builder.compute("gross_margin", "gross_profit / revenue")

builder.with_meta("author", "Finance Team")
builder.with_meta("version", "1.0")

model = builder.build()

print(f"Model ID: {model.id}")
print(f"Periods: {len(model.periods)} total")
print(f"Nodes: {len(list(model.nodes.keys()))} total")
print()

print("Period Breakdown:")
for period in model.periods:
    period_type = "Actual  " if period.is_actual else "Forecast"
    print(f"  {period.id} | {period_type} | {period.start} to {period.end}")


Model ID: Acme Corp Q1-Q4 2025
Periods: 4 total
Nodes: 6 total

Period Breakdown:
  2025Q1 | Actual   | 2025-01-01 to 2025-04-01
  2025Q2 | Actual   | 2025-04-01 to 2025-07-01
  2025Q3 | Forecast | 2025-07-01 to 2025-10-01
  2025Q4 | Forecast | 2025-10-01 to 2026-01-01


## Example 2 – Model Evaluation

Add forecasts to revenue/opex, evaluate the model, and display period-by-period results, including EBITDA margin.


In [9]:
builder = ModelBuilder.new("Evaluation Example")
builder.periods("2025Q1..Q4", "2025Q2")

builder.value(
    "revenue",
    [
        (PeriodId.quarter(2025, 1), AmountOrScalar.scalar(1_000_000.0)),
        (PeriodId.quarter(2025, 2), AmountOrScalar.scalar(1_100_000.0)),
    ],
)
builder.forecast("revenue", ForecastSpec.growth(0.05))

builder.value(
    "opex",
    [
        (PeriodId.quarter(2025, 1), AmountOrScalar.scalar(300_000.0)),
        (PeriodId.quarter(2025, 2), AmountOrScalar.scalar(320_000.0)),
    ],
)
builder.forecast("opex", ForecastSpec.forward_fill())

builder.compute("cogs", "revenue * 0.55")
builder.compute("gross_profit", "revenue - cogs")
builder.compute("ebitda", "gross_profit - opex")
builder.compute("ebitda_margin", "ebitda / revenue")

model = builder.build()

evaluator = Evaluator.new()
results = evaluator.evaluate(model)

print(f"Evaluation completed in {results.meta.eval_time_ms}ms")
print(f"Nodes evaluated: {results.meta.num_nodes}")
print(f"Periods evaluated: {results.meta.num_periods}")

print("\nResults by Period:")
for period in model.periods:
    print(f"\n{period.id} ({'Actual' if period.is_actual else 'Forecast'}):")
    print(f"  Revenue:        ${results.get('revenue', period.id):,.0f}")
    print(f"  COGS:           ${results.get('cogs', period.id):,.0f}")
    print(f"  Gross Profit:   ${results.get('gross_profit', period.id):,.0f}")
    print(f"  OpEx:           ${results.get('opex', period.id):,.0f}")
    print(f"  EBITDA:         ${results.get('ebitda', period.id):,.0f}")
    print(f"  EBITDA Margin:  {results.get('ebitda_margin', period.id):.1%}")


Evaluation completed in 0ms
Nodes evaluated: 6
Periods evaluated: 4

Results by Period:

2025Q1 (Actual):
  Revenue:        $1,000,000
  COGS:           $550,000
  Gross Profit:   $450,000
  OpEx:           $300,000
  EBITDA:         $150,000
  EBITDA Margin:  15.0%

2025Q2 (Actual):
  Revenue:        $1,100,000
  COGS:           $605,000
  Gross Profit:   $495,000
  OpEx:           $320,000
  EBITDA:         $175,000
  EBITDA Margin:  15.9%

2025Q3 (Forecast):
  Revenue:        $1,155,000
  COGS:           $635,250
  Gross Profit:   $519,750
  OpEx:           $320,000
  EBITDA:         $199,750
  EBITDA Margin:  17.3%

2025Q4 (Forecast):
  Revenue:        $1,212,750
  COGS:           $667,012
  Gross Profit:   $545,738
  OpEx:           $320,000
  EBITDA:         $225,738
  EBITDA Margin:  18.6%


## Example 3 – Dynamic Metric Registry

Load the built-in metric registry, inspect available metrics, and drill into `fin.gross_margin`.


In [10]:
registry = Registry.new()
registry.load_builtins()

print(f"Total built-in metrics loaded: {len(registry.list_metrics(None))}")

fin_metrics = registry.list_metrics("fin")
print(f"Metrics in 'fin' namespace: {len(fin_metrics)}")

metric = registry.get("fin.gross_margin")
print(f"Metric: {metric.name}")
print(f"  ID: {metric.id}")
print(f"  Formula: {metric.formula}")
print(f"  Category: {metric.category}")
print(f"  Requires: {metric.requires}")

print("\nSample built-in metrics:")
for metric_id in sorted(fin_metrics)[:10]:
    entry = registry.get(metric_id)
    print(f"  {metric_id}: {entry.name}")


Total built-in metrics loaded: 22
Metrics in 'fin' namespace: 22
Metric: Gross Margin %
  ID: gross_margin
  Formula: gross_profit / revenue
  Category: margins
  Requires: ['revenue', 'gross_profit']

Sample built-in metrics:
  fin.cogs_as_pct_revenue: COGS as % of Revenue
  fin.debt_service_coverage: Debt Service Coverage
  fin.debt_to_assets: Debt to Assets
  fin.debt_to_ebitda: Debt to EBITDA
  fin.debt_to_equity: Debt to Equity
  fin.ebit: EBIT
  fin.ebitda: EBITDA
  fin.ebitda_margin: EBITDA Margin %
  fin.ebt: EBT
  fin.equity_multiplier: Equity Multiplier


## Example 4 – Extension System

Build a minimal model, evaluate it, and initialize the extension registry (Corkscrew + Credit Scorecard entries).


In [11]:
builder = ModelBuilder.new("Extension Test")
builder.periods("2025Q1..Q2", None)

builder.value(
    "test_node",
    [
        (PeriodId.quarter(2025, 1), AmountOrScalar.scalar(100.0)),
        (PeriodId.quarter(2025, 2), AmountOrScalar.scalar(200.0)),
    ],
)

model = builder.build()

evaluator = Evaluator.new()
results = evaluator.evaluate(model)

ext_registry = ExtensionRegistry.new()

print("Extension system initialized")
print("\u2713 CorkscrewExtension available")
print("\u2713 CreditScorecardExtension available")


Extension system initialized
✓ CorkscrewExtension available
✓ CreditScorecardExtension available


## Example 5 – JSON Serialization

Serialize a model to JSON, then restore it with `FinancialModelSpec.from_json` to verify round-tripping.


In [12]:
builder = ModelBuilder.new("Serialization Test")
builder.periods("2025Q1..Q2", None)

builder.value(
    "revenue",
    [(PeriodId.quarter(2025, 1), AmountOrScalar.scalar(1000.0))],
)
builder.compute("profit", "revenue * 0.2")

model = builder.build()

json_str = model.to_json()
print("Model serialized to JSON successfully")
print(f"JSON length: {len(json_str)} bytes")

from finstack.statements.types import FinancialModelSpec

restored = FinancialModelSpec.from_json(json_str)
print(f"Model restored from JSON: {restored.id}")
print(f"Periods: {len(restored.periods)}")
print(f"Nodes: {len(list(restored.nodes.keys()))}")


Model serialized to JSON successfully
JSON length: 381 bytes
Model restored from JSON: Serialization Test
Periods: 2
Nodes: 2


All five examples are now executed interactively. Rerun individual cells as needed to tweak inputs or explore additional metrics.
